# NBA Pipeline Smoke Test

Loads curated Parquet via DuckDB views — no live `nba_api` calls.

Run a refresh first:
```bash
cd nba_pipeline
python -m nba_pipeline refresh --all
```

In [1]:
from pathlib import Path
import sys

# Allow importing the package when the notebook cwd is nba_pipeline/notebooks
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from nba_pipeline.config import Settings
from nba_pipeline.load.duckdb_views import connect

cfg = Settings(project_root=ROOT)
con = connect(cfg)
print("Curated root:", cfg.curated_dir)

Curated root: /Volumes/Data-700GB/Basketball DS/nba_pipeline/data/curated


In [2]:
print(con.execute("SHOW TABLES").fetchdf())
print("players:", con.execute("SELECT COUNT(*) FROM players").fetchone()[0])
print("teams:", con.execute("SELECT COUNT(*) FROM teams").fetchone()[0])
print("player_season_stats:", con.execute("SELECT COUNT(*) FROM player_season_stats").fetchone()[0])
print("shot_charts:", con.execute("SELECT COUNT(*) FROM shot_charts").fetchone()[0])

                  name
0  player_season_stats
1              players
2          shot_charts
3                teams
players: 5103
teams: 30
player_season_stats: 2867
shot_charts: 1035473


In [11]:
stats = con.execute("""
SELECT season, PLAYER_NAME, MIN, PTS, AST, REB, STL, BLK, TOV, FG3A, FTA
FROM player_season_stats
WHERE PLAYER_NAME IN ('LeBron James', 'Joel Embiid', 'Jaylen Brown', 'Tyrese Maxey')
ORDER BY PTS DESC
""").fetchdf()
stats

,season,PLAYER_NAME,MIN,PTS,AST,REB,STL,BLK,TOV,FG3A,FTA
0,2023-24,Joel Embiid,1309.371667,49.2,8.0,15.6,1.7,2.4,5.5,5.1,16.4
1,2022-23,Joel Embiid,2284.106667,47.1,5.9,14.4,1.4,2.4,4.9,4.3,16.6
2,2021-22,Joel Embiid,2296.405000,44.8,6.1,17.2,1.7,2.1,4.6,5.4,17.3
3,2025-26,Jaylen Brown,2442.838333,41.0,7.3,9.9,1.4,0.5,5.2,8.1,10.8
4,2025-26,Joel Embiid,1200.638333,40.5,5.9,11.6,0.8,1.8,4.3,6.3,13.3
5,2021-22,LeBron James,2084.335000,38.7,8.0,10.5,1.7,1.3,4.5,10.2,7.7
6,2022-23,LeBron James,1953.940000,37.8,8.9,10.9,1.2,0.8,4.2,9.0,7.8
7,2024-25,Joel Embiid,574.393333,37.7,7.1,12.9,1.2,1.5,5.2,6.4,14.1
8,2022-23,Jaylen Brown,2404.918333,35.5,4.6,9.1,1.5,0.5,3.9,9.7,6.8
9,2025-26,Tyrese Maxey,2661.118333,35.5,8.3,5.2,2.3,1.0,3.1,10.8,7.5


In [3]:
# Archetype clustering inputs: per-100 stats, minutes filter
stats = con.execute("""
SELECT season, PLAYER_NAME, MIN, PTS, AST, REB, STL, BLK, TOV, FG3A, FTA
FROM player_season_stats
WHERE MIN >= 500
ORDER BY PTS DESC
LIMIT 10
""").fetchdf()
stats

,season,PLAYER_NAME,MIN,PTS,AST,REB,STL,BLK,TOV,FG3A,FTA
0,2023-24,Joel Embiid,1309.371667,49.2,8.0,15.6,1.7,2.4,5.5,5.1,16.4
1,2022-23,Joel Embiid,2284.106667,47.1,5.9,14.4,1.4,2.4,4.9,4.3,16.6
2,2025-26,Giannis Antetokounmpo,1039.230000,45.9,9.1,16.3,1.6,1.1,5.3,2.1,16.5
3,2024-25,Shai Gilgeous-Alexander,2597.630000,45.0,8.8,6.9,2.4,1.4,3.3,7.9,12.1
4,2021-22,Joel Embiid,2296.405000,44.8,6.1,17.2,1.7,2.1,4.6,5.4,17.3
5,2022-23,Giannis Antetokounmpo,2023.615000,44.7,8.2,16.9,1.2,1.2,5.6,3.9,17.6
6,2025-26,Luka Dončić,2288.601667,44.6,11.0,10.3,2.2,0.7,5.3,14.4,13.4
7,2025-26,Shai Gilgeous-Alexander,2258.946667,44.1,9.3,6.1,2.0,1.1,3.1,6.2,12.8
8,2022-23,Luka Dončić,2390.471667,43.9,10.9,11.7,1.8,0.7,4.8,11.1,14.2
9,2023-24,Luka Dončić,2624.040000,42.9,12.4,11.7,1.8,0.7,5.1,13.5,11.0


In [16]:
# Shot chart sample (e.g. high-volume shooters)
shots = con.execute("""
SELECT season, PLAYER_NAME
    , avg(SHOT_DISTANCE) AS avg_shot_distance
    ,sum(shot_distance) AS total_shot_distance
    , sum(min) AS total_minutes
---, SHOT_MADE_FLAG, SHOT_DISTANCE, LOC_X, LOC_Y
FROM shot_charts
join player_season_stats USING (PLAYER_NAME, season)
WHERE SHOT_MADE_FLAG = 1
-- PLAYER_NAME LIKE '%Curry%'
AND SHOT_DISTANCE < 5 
AND MIN >= 500
GROUP BY season, PLAYER_NAME
ORDER BY 3 DESC
LIMIT 20
""").fetchdf()
shots

,season,PLAYER_NAME,avg_shot_distance,total_shot_distance,total_minutes
0,2023-24,Brice Sensabaugh,2.882353,49.0,9934.176667
1,2023-24,Jacob Gilyard,2.666667,8.0,2095.215000
2,2021-22,Joe Ingles,2.545455,56.0,24686.970000
3,2024-25,Elfrid Payton,2.538462,33.0,6640.465000
4,2023-24,Chris Paul,2.470588,42.0,26025.130000
5,2021-22,Klay Thompson,2.469388,121.0,46101.731667
6,2021-22,Theo Maledon,2.428571,102.0,38145.240000
7,2021-22,Mike Conley,2.409091,159.0,135837.130000
8,2023-24,Caleb Houstan,2.400000,12.0,4076.675000
9,2023-24,Stephen Curry,2.390411,349.0,353544.596667


In [17]:
shots = con.execute("""
SELECT season, PLAYER_NAME, SHOT_MADE_FLAG, SHOT_DISTANCE, LOC_X, LOC_Y
FROM shot_charts
WHERE PLAYER_NAME LIKE '%Curry%'
LIMIT 20
""").fetchdf()
shots

,season,PLAYER_NAME,SHOT_MADE_FLAG,SHOT_DISTANCE,LOC_X,LOC_Y
0,2021-22,Stephen Curry,0,28,-109,260
1,2021-22,Stephen Curry,0,26,48,257
2,2021-22,Stephen Curry,1,25,-165,189
3,2021-22,Stephen Curry,0,1,-13,12
4,2021-22,Stephen Curry,0,2,-15,22
5,2021-22,Stephen Curry,0,2,18,16
6,2021-22,Stephen Curry,1,2,-7,29
7,2021-22,Stephen Curry,0,26,135,228
8,2021-22,Stephen Curry,0,2,-9,18
9,2021-22,Stephen Curry,1,1,-1,10


In [19]:
import pandas as pd

In [20]:
# Database-wide summary
tables = con.execute("SHOW TABLES").fetchdf().iloc[:, 0].tolist()

table_summary = []
date_summary = []

for table in tables:
    columns = con.execute(f'DESCRIBE "{table}"').fetchdf()["column_name"].tolist()
    column_map = {c.lower(): c for c in columns}

    row_count = con.execute(f'SELECT COUNT(*) FROM "{table}"').fetchone()[0]
    table_summary.append({"table": table, "rows": row_count, "columns": len(columns)})

    date_columns = [
        column_map[c] for c in column_map
        if "date" in c or c in {"event_date", "game_date"}
    ]

    for date_column in date_columns:
        result = con.execute(
            f'''
            SELECT MIN("{date_column}"), MAX("{date_column}")
            FROM "{table}"
            '''
        ).fetchone()

        date_summary.append({
            "table": table,
            "date_column": date_column,
            "min_date": result[0],
            "max_date": result[1],
        })

print("Tables:")
display(pd.DataFrame(table_summary))

print("Date ranges:")
display(pd.DataFrame(date_summary))

# Overall database counts
count_summary = []

for table, column, label in [
    ("games", "GAME_ID", "total_games"),
    ("teams", "TEAM_ID", "total_teams"),
    ("players", "PLAYER_ID", "total_players"),
]:
    if table in tables:
        columns = con.execute(f'DESCRIBE "{table}"').fetchdf()["column_name"].str.upper().tolist()
        if column in columns:
            value = con.execute(
                f'SELECT COUNT(DISTINCT "{column}") FROM "{table}"'
            ).fetchone()[0]
            count_summary.append({"metric": label, "value": value})

display(pd.DataFrame(count_summary))

# Team-level points and minutes, when available in a table
team_table = "player_season_stats"

if team_table in tables:
    columns = con.execute(f'DESCRIBE "{team_table}"').fetchdf()["column_name"].tolist()
    column_map = {c.lower(): c for c in columns}

    team_column = next(
        (column_map[c] for c in ["team_name", "team_abbreviation", "team_id"] if c in column_map),
        None,
    )

    if team_column and "pts" in column_map and "min" in column_map:
        team_totals = con.execute(
            f'''
            SELECT
                season,
                "{team_column}" AS team,
                SUM(PTS) AS total_points,
                SUM(MIN) AS total_minutes
            FROM "{team_table}"
            GROUP BY season, "{team_column}"
            ORDER BY season, total_points DESC
            '''
        ).fetchdf()

        display(team_totals)
    else:
        print("Team-level PTS/MIN columns were not found in player_season_stats.")

Tables:


,table,rows,columns
0,player_season_stats,2867,36
1,players,5103,5
2,shot_charts,1035473,12
3,teams,30,7


Date ranges:


,table,date_column,min_date,max_date
0,shot_charts,GAME_DATE,20211019,20260412


,metric,value
0,total_teams,30
1,total_players,5103


,season,team,total_points,total_minutes
0,2021-22,OKC,469.8,19852.795000
1,2021-22,IND,459.0,17958.618333
2,2021-22,CLE,456.7,20366.070000
3,2021-22,POR,448.3,19489.631667
4,2021-22,BOS,431.6,19118.551667
...,...,...,...,...
145,2025-26,NOP,352.3,18908.406667
146,2025-26,NYK,343.9,20635.816667
147,2025-26,HOU,336.9,19905.000000
148,2025-26,BOS,329.6,19363.978333
